# NetSentinel — Expert 5: Reconnaissance / Port Scan Detector (XGBoost)

**Model**: XGBoost Gradient-Boosted Trees (binary: PortScan vs Benign)  
**Datasets**: CIC-IDS2017, LITNET-2020, UNSW-NB15, CSE-CIC-IDS2018  
**Task**: Binary classification (Port Scan vs Benign)  
**Key Innovation**: Multi-dataset training for cross-environment generalization  
**Export**: ONNX  

---

## Datasets to attach in Kaggle (right sidebar → Add Input)

| # | Kaggle dataset path | What it provides |
|---|---|---|
| 1 | `cicdataset/cicids2017` or `dhoogla/cicids2017` | CIC-IDS2017 — Thursday PortScan + all-day benign |
| 2 | `rashidthihan/preprocessed-litnet-2020-dataset` | LITNET-2020 — real university network scans |
| 3 | `mrwellsdavid/unsw-nb15` | UNSW-NB15 — Reconnaissance/Generic attacks |
| 4 | `dhoogla/csecicids2018` | CSE-CIC-IDS2018 — Infiltration/Scans |

**GPU not required** — XGBoost trains on CPU in ~5-10 minutes.

In [ ]:
!pip install -q onnxruntime xgboost shap skl2onnx onnxmltools

In [ ]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, accuracy_score, precision_score, recall_score,
    precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Ready.')

## 1. Data Loading — CIC-IDS2017

The Thursday afternoon capture contains labeled **PortScan** traffic. We also load benign traffic from all other days for the negative class.

In [ ]:
# ============================================================
# Scan for CSV files across all 4 datasets
# ============================================================

DATA_ROOT = '/kaggle/input'

# Expected dataset paths (attach these in Kaggle sidebar)
DATASET_PATHS = {
    'CIC-IDS2017': [
        '/kaggle/input/cicids2017',
        '/kaggle/input/datasets/cicdataset/cicids2017',
        '/kaggle/input/datasets/dhoogla/cicids2017',
    ],
    'LITNET-2020': [
        '/kaggle/input/preprocessed-litnet-2020-dataset',
        '/kaggle/input/datasets/rashidthihan/preprocessed-litnet-2020-dataset',
    ],
    'UNSW-NB15': [
        '/kaggle/input/unsw-nb15',
        '/kaggle/input/datasets/mrwellsdavid/unsw-nb15',
    ],
    'CSE-CIC-IDS2018': [
        '/kaggle/input/csecicids2018',
        '/kaggle/input/datasets/dhoogla/csecicids2018',
    ],
}

# Find which datasets are actually present
found_datasets = {}
for name, candidates in DATASET_PATHS.items():
    for path in candidates:
        if os.path.isdir(path):
            found_datasets[name] = path
            break

print(f'Found {len(found_datasets)}/4 datasets:')
for name, path in found_datasets.items():
    print(f'  [OK] {name:20s} -> {path}')

missing = set(DATASET_PATHS.keys()) - set(found_datasets.keys())
if missing:
    print(f'  [!!] Missing: {missing} -- attach in Kaggle sidebar')

# Scan all CSVs
csv_files = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.csv'), recursive=True))
print(f'\nFound {len(csv_files)} total CSV files:')
for f in csv_files:
    sz = os.path.getsize(f) / (1024*1024)
    print(f'  {f:70s} {sz:8.1f} MB')


In [ ]:
# ============================================================
# Load all CSVs, align columns, and concatenate
# ============================================================

def identify_dataset(filepath):
    """Tag each file with its source dataset for provenance."""
    fp = filepath.lower().replace('\\', '/')
    if 'litnet' in fp: return 'LITNET-2020'
    if 'unsw' in fp: return 'UNSW-NB15'
    if 'cic' in fp and '2018' in fp: return 'CSE-CIC-IDS2018'
    if 'csecic' in fp: return 'CSE-CIC-IDS2018'
    return 'CIC-IDS2017'  # default

def standardize_columns(df):
    col_map = {}
    for c in df.columns:
        c_clean = c.strip().lower().replace(' ', '').replace('_', '')
        
        # Label mapping (covers all 4 datasets)
        if c_clean in ['label', 'attackcat', 'attack', 'attackcategory', 'class',
                       'attacktype', 'category', 'labelcat']:
            col_map[c] = 'target_label'
        # Core features mapping
        elif c_clean in ['flowduration', 'dur', 'duration']:
            col_map[c] = 'flow_duration'
        elif c_clean in ['protocol', 'proto', 'service']:
            col_map[c] = 'protocol'
        elif c_clean in ['sourceport', 'sport', 'srcport']:
            col_map[c] = 'src_port'
        elif c_clean in ['destinationport', 'dport', 'dstport']:
            col_map[c] = 'dst_port'
        elif c_clean in ['totalfwdpackets', 'spkts', 'srcpkts']:
            col_map[c] = 'src_pkts'
        elif c_clean in ['totalbackwardpackets', 'dpkts', 'dstpkts']:
            col_map[c] = 'dst_pkts'
        elif c_clean in ['totallengthoffwdpackets', 'sbytes', 'srcbytes']:
            col_map[c] = 'src_bytes'
        elif c_clean in ['totallengthofbwdpackets', 'dbytes', 'dstbytes']:
            col_map[c] = 'dst_bytes'
        else:
            col_map[c] = c_clean  # fallback to normalized name
            
    df = df.rename(columns=col_map)
    df = df.loc[:, ~df.columns.duplicated()]
    return df

dfs = []
dataset_stats = {}
for f in csv_files:
    try:
        df = pd.read_csv(f, low_memory=False, encoding='utf-8')
        df = standardize_columns(df)
        src = identify_dataset(f)
        df['_source'] = src
        dfs.append(df)
        dataset_stats[src] = dataset_stats.get(src, 0) + len(df)
        print(f'  OK: [{src:18s}] {os.path.basename(f):45s} -> {len(df):>10,} rows, {len(df.columns)} cols')
    except Exception as e:
        try:
            df = pd.read_csv(f, low_memory=False, encoding='latin-1')
            df = standardize_columns(df)
            src = identify_dataset(f)
            df['_source'] = src
            dfs.append(df)
            dataset_stats[src] = dataset_stats.get(src, 0) + len(df)
            print(f'  OK (latin-1): [{src:14s}] {os.path.basename(f):38s} -> {len(df):>10,} rows')
        except Exception as e2:
            print(f'  ERR: {os.path.basename(f)}: {str(e2)[:80]}')

data = pd.concat(dfs, ignore_index=True)
print(f'\nTotal: {len(data):,} rows, {len(data.columns)} columns')
print(f'\nRows per dataset:')
for name, count in sorted(dataset_stats.items()):
    print(f'  {name:20s} {count:>12,} rows')



In [ ]:
# ============================================================
# Find label column and explore labels
# ============================================================

label_col = 'target_label'
assert label_col in data.columns, 'Unified label column not found!'

print(f'Label column: "{label_col}"')
print(f'\nLabel distribution:')
print(data[label_col].value_counts())



## 2. Binary Label Construction: PortScan vs Benign

We keep only **PortScan** and **BENIGN** rows, creating a clean binary classification task.

In [ ]:
# ============================================================
# Filter to PortScan/Reconnaissance + Benign only
# ============================================================

data[label_col] = data[label_col].astype(str).str.strip()
labels_lower = data[label_col].str.lower()

# Match benign labels across all 4 datasets:
# CIC-IDS2017:      'BENIGN'
# LITNET-2020:       'Normal' or 'Benign'
# UNSW-NB15:         'Normal'
# CSE-CIC-IDS2018:   'Benign'
is_benign = labels_lower.str.contains(
    'benign|normal', na=False
) & ~labels_lower.str.contains('abnormal', na=False)  # exclude 'abnormal'

# Match portscan/reconnaissance labels across all 4 datasets:
# CIC-IDS2017:      'PortScan'
# LITNET-2020:       'Port scanning', 'Scanning'
# UNSW-NB15:         'Reconnaissance', 'Generic' (recon-related)
# CSE-CIC-IDS2018:   'Infiltration', 'PortScan'
is_portscan = labels_lower.str.contains(
    'portscan|port scan|port_scan|scanning|reconnaissance|recon|infiltra',
    na=False
)

data_filtered = data[is_benign | is_portscan].copy()
print(f'Benign:    {is_benign.sum():>10,}')
print(f'PortScan:  {is_portscan.sum():>10,}')
print(f'Kept:      {len(data_filtered):>10,} / {len(data):,}')
print(f'Dropped:   {len(data) - len(data_filtered):>10,} (other attack types)')

# Binary label: 0 = Benign, 1 = PortScan
data_filtered['target'] = (
    ~data_filtered[label_col].str.lower().str.contains('benign|normal')
    | data_filtered[label_col].str.lower().str.contains('abnormal')
).astype(int)

print(f'\nBinary label distribution:')
print(data_filtered['target'].value_counts().rename({0: 'Benign', 1: 'PortScan'}))

# Per-dataset breakdown
if '_source' in data_filtered.columns:
    print(f'\nPer-dataset contribution:')
    for src in sorted(data_filtered['_source'].unique()):
        mask = data_filtered['_source'] == src
        b = (data_filtered.loc[mask, 'target'] == 0).sum()
        p = (data_filtered.loc[mask, 'target'] == 1).sum()
        print(f'  {src:20s}  Benign={b:>10,}  PortScan={p:>10,}')



## 3. Feature Engineering & Cleaning

In [ ]:
# ============================================================
# Drop non-feature columns
# ============================================================

drop_cols = []
for c in data_filtered.columns:
    cl = c.strip().lower().replace(' ', '').replace('_', '')
    if any(kw in cl for kw in [
        'label', 'class', 'attack', 'category', 'target',
        'flowid', 'srcip', 'sourceip', 'dstip', 'destip', 'destinationip',
        'srcport', 'sourceport', 'dstport', 'destport', 'destinationport',
        'timestamp', 'datetime', 'source'
    ]):
        drop_cols.append(c)

# Also drop the _source tracking column
if '_source' in data_filtered.columns and '_source' not in drop_cols:
    drop_cols.append('_source')

print(f'Dropping {len(drop_cols)} non-feature columns: {drop_cols}')
y = data_filtered['target'].values
X = data_filtered.drop(columns=drop_cols, errors='ignore')

# Keep only numeric columns
X = X.apply(pd.to_numeric, errors='coerce')

# Replace inf and drop all-NaN columns
X = X.replace([np.inf, -np.inf], np.nan)
nan_frac = X.isna().mean()
bad_cols = nan_frac[nan_frac > 0.5].index.tolist()
if bad_cols:
    print(f'Dropping {len(bad_cols)} columns with >50% NaN: {bad_cols}')
    X = X.drop(columns=bad_cols)

# Fill remaining NaNs with column median
X = X.fillna(X.median())

# Final clean: drop any remaining all-NaN columns and zero-variance
X = X.dropna(axis=1, how='all')
zero_var = X.columns[X.std() == 0].tolist()
if zero_var:
    print(f'Dropping {len(zero_var)} zero-variance columns: {zero_var}')
    X = X.drop(columns=zero_var)

# Align y with X (drop rows that became NaN)
valid_mask = X.notna().all(axis=1)
X = X[valid_mask]
y = y[valid_mask.values]

feature_names = list(X.columns)
print(f'\nFinal: {X.shape[0]:,} samples × {X.shape[1]} features')
print(f'Class balance: Benign={int((y==0).sum()):,}, PortScan={int((y==1).sum()):,}')

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')
print(f'Train balance: Benign={int((y_train==0).sum()):,}, PortScan={int((y_train==1).sum()):,}')
print(f'Test  balance: Benign={int((y_test==0).sum()):,}, PortScan={int((y_test==1).sum()):,}')

## 5. XGBoost Training with Eval Logging

In [ ]:
# ============================================================
# Train XGBoost binary classifier
# ============================================================

# Class weight for imbalanced data
n_benign = int((y_train == 0).sum())
n_scan = int((y_train == 1).sum())
scale_pos_weight = n_benign / max(n_scan, 1)
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

params = {
    'objective': 'binary:logistic',
    'eval_metric': ['logloss', 'auc', 'error'],
    'max_depth': 8,
    'learning_rate': 0.1,
    'n_estimators': 300,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'tree_method': 'hist',
    'random_state': SEED,
    'verbosity': 0,
}

model = xgb.XGBClassifier(**params)

start_time = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)
train_time = time.time() - start_time
print(f'\nTraining time: {train_time:.1f} seconds')

## 6. Training Curves — Loss, AUC, Accuracy

In [ ]:
# ============================================================
# Plot training curves
# ============================================================

results = model.evals_result()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(results['validation_0']['logloss'], label='Train', linewidth=2)
axes[0].plot(results['validation_1']['logloss'], label='Validation', linewidth=2)
axes[0].set_title('Log Loss', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Boosting Round')
axes[0].legend()
axes[0].grid(alpha=0.3)

# AUC
axes[1].plot(results['validation_0']['auc'], label='Train', linewidth=2, color='green')
axes[1].plot(results['validation_1']['auc'], label='Validation', linewidth=2, color='darkgreen')
axes[1].set_title('AUC-ROC', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Boosting Round')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Error rate (1 - accuracy)
train_acc = [1 - e for e in results['validation_0']['error']]
val_acc   = [1 - e for e in results['validation_1']['error']]
axes[2].plot(train_acc, label='Train', linewidth=2, color='purple')
axes[2].plot(val_acc, label='Validation', linewidth=2, color='orchid')
axes[2].set_title('Accuracy', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Boosting Round')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Expert 5: Port Scan Detector (XGBoost) — Training Curves',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'expert5_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation — Classification Report + Confusion Matrix

In [ ]:
# ============================================================
# Evaluate on test set
# ============================================================

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print('=' * 55)
print('  EXPERT 5: PORT SCAN DETECTOR — RESULTS')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=['Benign', 'PortScan']))
print(f'Accuracy:  {acc:.6f}')
print(f'Precision: {prec:.6f}')
print(f'Recall:    {rec:.6f}')
print(f'F1-Score:  {f1:.6f}')
print(f'AUC-ROC:   {auc:.6f}')

In [ ]:
# ============================================================
# Confusion Matrix
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['Benign', 'PortScan'],
            yticklabels=['Benign', 'PortScan'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix (F1: {f1:.4f})', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Normalized (percentages)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['Benign', 'PortScan'],
            yticklabels=['Benign', 'PortScan'], ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix (%)', fontweight='bold')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle('Expert 5: Port Scan Detector', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'expert5_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Importance + SHAP Explainability

In [ ]:
# ============================================================
# Feature importance (gain)
# ============================================================

importances = model.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.head(20).plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 20 Features — Port Scan Detector', fontweight='bold', fontsize=13)
ax.set_xlabel('Feature Importance (Gain)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'expert5_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
for i, (name, val) in enumerate(feat_imp.head(10).items(), 1):
    print(f'  {i:2d}. {name:40s} {val:.4f}')

In [ ]:
# ============================================================
# SHAP analysis (on a sample for speed)
# ============================================================

import shap

sample_size = min(5000, len(X_test))
X_shap = pd.DataFrame(X_test[:sample_size], columns=feature_names)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)

# Beeswarm plot
fig = plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
plt.title('SHAP Feature Impact — Port Scan Detector', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'expert5_shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# Waterfall for a single PortScan prediction
scan_idx = np.where(y_test[:sample_size] == 1)[0]
if len(scan_idx) > 0:
    fig = plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[scan_idx[0]],
        base_values=explainer.expected_value,
        data=X_shap.iloc[scan_idx[0]],
        feature_names=feature_names
    ), max_display=12, show=False)
    plt.title('SHAP Waterfall — Single Port Scan Alert', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'expert5_shap_waterfall.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 9. Export to ONNX

In [ ]:
# ============================================================
# Export XGBoost model to ONNX
# ============================================================

from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort

n_features = len(feature_names)
initial_type = [('features', FloatTensorType([None, n_features]))]
onnx_model = convert_xgboost(model, initial_types=initial_type, target_opset=14)

onnx_path = os.path.join(OUTPUT_DIR, 'expert5_portscan_xgboost.onnx')
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f'ONNX saved: {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

# Verify ONNX inference
sess = ort.InferenceSession(onnx_path)
input_name = sess.get_inputs()[0].name
test_input = X_test[:5].astype(np.float32)
onnx_out = sess.run(None, {input_name: test_input})
print(f'ONNX input name: {input_name}')
print(f'ONNX output labels: {onnx_out[0][:5]}')
print(f'ONNX output probs:  {onnx_out[1][:5]}')

# Compare with native XGBoost
native_pred = model.predict(test_input)
print(f'Native predictions:  {native_pred[:5]}')
print(f'Match: {np.array_equal(onnx_out[0][:5], native_pred[:5])}')

In [ ]:
# ============================================================
# Inference benchmark
# ============================================================

import timeit

single_sample = X_test[:1].astype(np.float32)
batch_sample = X_test[:256].astype(np.float32)

# Single sample latency
n_runs = 1000
t_single = timeit.timeit(
    lambda: sess.run(None, {input_name: single_sample}),
    number=n_runs
)

# Batch latency
t_batch = timeit.timeit(
    lambda: sess.run(None, {input_name: batch_sample}),
    number=n_runs
)

latency_single = t_single / n_runs * 1000  # ms
latency_batch = t_batch / n_runs * 1000    # ms
throughput = 1000 / latency_single          # samples/sec

print(f'Single sample: {latency_single:.4f} ms')
print(f'Batch (256):   {latency_batch:.4f} ms ({latency_batch/256*1000:.2f} μs/sample)')
print(f'Throughput:    {throughput:,.0f} flows/sec')

## 10. Model Card & Metrics Export

In [ ]:
# ============================================================
# Save metadata and feature names
# ============================================================

metrics = {
    'model_name': 'NetSentinel Expert 5: Port Scan Detector',
    'model_type': 'XGBoost Gradient-Boosted Trees',
    'version': '1.0.0',
    'task': 'Binary Classification (PortScan vs Benign)',
    'accuracy': float(acc),
    'precision': float(prec),
    'recall': float(rec),
    'f1': float(f1),
    'auc_roc': float(auc),
    'training_time_seconds': float(train_time),
    'n_features': n_features,
    'dataset': {
        'sources': ['CIC-IDS2017', 'LITNET-2020', 'UNSW-NB15', 'CSE-CIC-IDS2018'],
        'total_samples': int(len(y)),
        'train_samples': int(len(y_train)),
        'test_samples': int(len(y_test)),
    },
    'inference': {
        'format': 'ONNX',
        'latency_ms': float(latency_single),
        'throughput_per_sec': float(throughput),
    },
    'top_10_features': feat_imp.head(10).index.tolist(),
    'mitre_mapping': {
        'T1046': 'Network Service Discovery',
        'T1595': 'Active Scanning',
        'T1595.001': 'Scanning IP Blocks',
        'T1595.002': 'Vulnerability Scanning',
    }
}

with open(os.path.join(OUTPUT_DIR, 'expert5_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, 'expert5_feature_names.json'), 'w') as f:
    json.dump(feature_names, f, indent=2)

# ============================================================
# Print Model Card
# ============================================================

print('\n' + '=' * 55)
print('  MODEL CARD: Port Scan Detector (Expert 5)')
print('=' * 55)
print(f'  Arch:       XGBoost (max_depth={params["max_depth"]}, n_est={params["n_estimators"]})')
print(f'  Datasets:   CIC-IDS2017 + LITNET-2020 + UNSW-NB15 + CSE-CIC-IDS2018 ({len(y):,} samples)')
print(f'  Features:   {n_features}')
print(f'  Accuracy:   {acc:.4f}')
print(f'  Precision:  {prec:.4f}')
print(f'  Recall:     {rec:.4f}')
print(f'  F1-Score:   {f1:.4f}')
print(f'  AUC-ROC:    {auc:.4f}')
print(f'  Latency:    {latency_single:.4f} ms ({throughput:,.0f} flows/sec)')
print(f'  ONNX:       {onnx_path}')
print(f'  MITRE:      T1046, T1595')
print('=' * 55)

In [ ]:
# ============================================================
# Package all outputs into a zip for download
# ============================================================

import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/netsentinel_portscan_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        fp = os.path.join(OUTPUT_DIR, f)
        zf.write(fp, f)
        print(f'  {f:45s} ({os.path.getsize(fp)/1024:.1f} KB)')

print(f'\nZip: {os.path.getsize(zip_path)/1024/1024:.1f} MB')
FileLink(zip_path)